In [1]:
import pandas as pd

from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print(Path.cwd())

RAW_FILE = "data/raw/Online Retail.xlsx"

df = pd.read_excel(RAW_FILE)

E:\Programming\Python\StockWise


In [2]:
# 1. Inspect Data

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDate range:")
print(df["InvoiceDate"].min())
print(df["InvoiceDate"].max())

print("\nUnique products:", df["StockCode"].nunique())
print("Unique invoices:", df["InvoiceNo"].nunique())

print("\nNegative quantities:", (df["Quantity"] < 0).sum())
print("Zero quantities:", (df["Quantity"] == 0).sum())

print("\nCancellation invoices:", df["InvoiceNo"].astype(str).str.startswith("C").sum())

Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

First 5 rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  

Data types:
InvoiceNo              ob

In [3]:
# 2. Data Quality

invoice = df["InvoiceNo"].astype(str).str.strip()
is_cancellation = invoice.str.startswith("C")
is_negative = df["Quantity"] < 0

print("Cancellation rows:", is_cancellation.sum())
print("Negative quantity rows:", is_negative.sum())

print("\nOverlap:")
print("Both:", (is_cancellation & is_negative).sum())
print("Cancellation with non-negative quantity:",
      (is_cancellation & ~is_negative).sum())
print("Negative quantity without cancellation:",
      (~is_cancellation & is_negative).sum())

print("\nNegative quantity examples:")
print(
    df.loc[is_negative,
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate"]]
    .head(10).to_string(index=False)
    )

print("\nCancellation examples:")
print(
    df.loc[is_cancellation, 
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate"]]
    .head(10).to_string(index=False)
    )

print("\nMissing descriptions by product:")
print(df.loc[df["Description"].isna(), "StockCode"].value_counts().head(10))


# Look at cancellation invoices and their possible original invoice numbers
cancellations = df[df["InvoiceNo"].astype(str).str.startswith("C")].copy()

cancellations["OriginalInvoiceNo"] = (
    cancellations["InvoiceNo"]
    .astype(str)
    .str.replace("^C", "", regex=True)
    )

# Check whether the invoice without "C" exists
all_invoices = set(df["InvoiceNo"].astype(str))

cancellations["OriginalExists"] = (
    cancellations["OriginalInvoiceNo"]
    .isin(all_invoices)
    )

print("\nCancellation invoice matching:")
print(cancellations["OriginalExists"].value_counts())

print("\nExamples:")
print(
    cancellations[
        [
            "InvoiceNo",
            "OriginalInvoiceNo",
            "StockCode",
            "Quantity",
            "OriginalExists",
        ]
        ].head(20).to_string(index=False)
    )


positive_qty = df.loc[df["Quantity"] > 0, "Quantity"].sum()
negative_qty = df.loc[df["Quantity"] < 0, "Quantity"].sum()


print("\nQuantity summary:")
print(f"Positive quantity: {positive_qty:,}")
print(f"Negative quantity: {negative_qty:,}")
print(f"Absolute negative quantity: {abs(negative_qty):,}")

return_rate = abs(negative_qty) / positive_qty

print(f"Overall return/cancellation quantity rate:",
      f"{return_rate:.2%}")

product_summary = (
    df.assign(
        PositiveQuantity=df["Quantity"].clip(lower=0),
        NegativeQuantity=df["Quantity"].clip(upper=0)
    )
    .groupby("StockCode")
    .agg(
        PositiveQuantity=("PositiveQuantity", "sum"),
        NegativeQuantity=("NegativeQuantity", "sum")
    )
)

product_summary["ReturnRate"] = (
    product_summary["NegativeQuantity"].abs()
    / product_summary["PositiveQuantity"]
    )

print("\nProducts with highest return rates:")
print(product_summary[
    product_summary["PositiveQuantity"] > 0
    ].sort_values("ReturnRate", ascending=False)
      .head(20).to_string())


problem_products = [
    "79323P",
    "79323B",
    "84614A",
    "84613C",
    "35610A"
]

print(df[df["StockCode"].isin(problem_products)][
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate"
        ]
    ].sort_values(["StockCode", "InvoiceDate"]).to_string(index=False))


negative = df[df["Quantity"] < 0].copy()

print("\nNegative transactions by description:")
print(negative["Description"].fillna("<MISSING>").value_counts().head(30).to_string())

print("\nNegative transactions without cancellation invoice:")
print(negative[~negative["InvoiceNo"].astype(str).str.startswith("C")][
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate"
        ]
    ].head(50).to_string(index=False))

Cancellation rows: 9288
Negative quantity rows: 10624

Overlap:
Both: 9288
Cancellation with non-negative quantity: 0
Negative quantity without cancellation: 1336

Negative quantity examples:
InvoiceNo StockCode  Quantity         InvoiceDate
  C536379         D        -1 2010-12-01 09:41:00
  C536383    35004C        -1 2010-12-01 09:49:00
  C536391     22556       -12 2010-12-01 10:24:00
  C536391     21984       -24 2010-12-01 10:24:00
  C536391     21983       -24 2010-12-01 10:24:00
  C536391     21980       -24 2010-12-01 10:24:00
  C536391     21484       -12 2010-12-01 10:24:00
  C536391     22557       -12 2010-12-01 10:24:00
  C536391     22553       -24 2010-12-01 10:24:00
  C536506     22960        -6 2010-12-01 12:38:00

Cancellation examples:
InvoiceNo StockCode  Quantity         InvoiceDate
  C536379         D        -1 2010-12-01 09:41:00
  C536383    35004C        -1 2010-12-01 09:49:00
  C536391     22556       -12 2010-12-01 10:24:00
  C536391     21984       -24 2010

In [4]:
# 3. Inspect Product

product = "23843"

print(f"Investigating product: {product}")

product_data = df[df["StockCode"].astype(str).str.strip() == product].copy()

print("\n--- Product Information ---")

print(product_data[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "Country",
        ]
    ].to_string(index=False)
)

print("\n--- Summary ---")

print(f"Number of transaction rows: {len(product_data):,}")
print(f"Total quantity: {product_data['Quantity'].sum():,}")
print(f"Positive quantity: {product_data.loc[product_data['Quantity'] > 0, 'Quantity'].sum():,}")
print(f"Negative quantity: {product_data.loc[product_data['Quantity'] < 0, 'Quantity'].sum():,}")

Investigating product: 23843

--- Product Information ---
InvoiceNo StockCode                 Description  Quantity         InvoiceDate  UnitPrice        Country
   581483     23843 PAPER CRAFT , LITTLE BIRDIE     80995 2011-12-09 09:15:00       2.08 United Kingdom
  C581484     23843 PAPER CRAFT , LITTLE BIRDIE    -80995 2011-12-09 09:27:00       2.08 United Kingdom

--- Summary ---
Number of transaction rows: 2
Total quantity: 0
Positive quantity: 80,995
Negative quantity: -80,995


In [5]:
# 4. Inspect Cancellations

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["StockCode"] = df["StockCode"].astype(str).str.strip()

# --------------------------------------------------
# Separate sales and cancellations
# --------------------------------------------------

sales = df[df["Quantity"] > 0].copy()
cancellations = df[df["InvoiceNo"].astype(str).str.startswith("C")].copy()

cancellations["CancelledQuantity"] = cancellations["Quantity"].abs()

print("--- Cancellation Investigation ---")
print(f"Sales rows: {len(sales):,}")
print(f"Cancellation rows: {len(cancellations):,}")

# --------------------------------------------------
# Prepare row IDs
# --------------------------------------------------

sales = sales.reset_index().rename(columns={"index": "SaleRowID"})
cancellations = cancellations.reset_index().rename(columns={"index": "CancellationRowID"})

# Sort chronologically
sales = sales.sort_values("InvoiceDate")
cancellations = cancellations.sort_values("InvoiceDate")

# --------------------------------------------------
# Find one-to-one matches
# --------------------------------------------------

used_sales = set()
matches = []

for _, cancellation in cancellations.iterrows():

    candidates = sales[
        (sales["StockCode"] == cancellation["StockCode"])
        & (sales["UnitPrice"] == cancellation["UnitPrice"])
        & (sales["Quantity"] == cancellation["CancelledQuantity"])
        & (sales["InvoiceDate"] < cancellation["InvoiceDate"])
        & (~sales["SaleRowID"].isin(used_sales))
    ]

    if candidates.empty:
        continue

    # Choose the closest previous sale
    candidates = candidates.copy()

    candidates["TimeDifference"] = (cancellation["InvoiceDate"] - candidates["InvoiceDate"])

    best = candidates.loc[candidates["TimeDifference"].idxmin()]

    used_sales.add(best["SaleRowID"])

    matches.append(
        {
            "CancellationRowID": cancellation["CancellationRowID"],
            "InvoiceNo_cancel": cancellation["InvoiceNo"],
            "InvoiceNo_sale": best["InvoiceNo"],
            "StockCode": cancellation["StockCode"],
            "Quantity_cancel": cancellation["Quantity"],
            "Quantity_sale": best["Quantity"],
            "UnitPrice": cancellation["UnitPrice"],
            "InvoiceDate_sale": best["InvoiceDate"],
            "InvoiceDate_cancel": cancellation["InvoiceDate"],
            "DaysDifference": (cancellation["InvoiceDate"] - best["InvoiceDate"]).total_seconds() / 86400,
        }
    )

matches = pd.DataFrame(matches)

# --------------------------------------------------
# Results
# --------------------------------------------------

print("\n--- Matching Results ---")

print(f"Matched cancellation rows: {len(matches):,}")
print(f"Unmatched cancellation rows: {len(cancellations) - len(matches):,}")
print(f"Cancellation rows with a matched previous sale: {len(matches):,}")

if not matches.empty:

    print("\n--- Example Matches ---")

    columns = [
        "InvoiceNo_cancel",
        "InvoiceNo_sale",
        "StockCode",
        "Quantity_cancel",
        "Quantity_sale",
        "UnitPrice",
        "InvoiceDate_sale",
        "InvoiceDate_cancel",
        "DaysDifference",
    ]

    print(matches[columns].head(20).to_string(index=False))

    print("\n--- Largest Time Differences ---")

    print(matches.sort_values("DaysDifference", ascending=False)[columns].head(20).to_string(index=False))

    print("\n--- Time Difference Statistics ---")

    print(
        matches["DaysDifference"].describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
    )

    print("\n--- Matches Within Time Windows ---")

    for days in [1, 7, 14, 30, 60, 90, 180, 365]:
        count = (matches["DaysDifference"] <= days).sum()

        print(f"Within {days:>3} days: {count:>5,} ({count / len(matches) * 100:.1f}%)")

--- Cancellation Investigation ---
Sales rows: 531,285
Cancellation rows: 9,288

--- Matching Results ---
Matched cancellation rows: 6,994
Unmatched cancellation rows: 2,294
Cancellation rows with a matched previous sale: 6,994

--- Example Matches ---
InvoiceNo_cancel  InvoiceNo_sale StockCode  Quantity_cancel  Quantity_sale  UnitPrice    InvoiceDate_sale  InvoiceDate_cancel  DaysDifference
         C536506          536466     22960               -6              6       4.25 2010-12-01 12:23:00 2010-12-01 12:38:00        0.010417
         C536543          536522     22632               -1              1       2.10 2010-12-01 12:49:00 2010-12-01 14:30:00        0.070139
         C536548          536385     22168               -2              2       8.50 2010-12-01 09:56:00 2010-12-01 14:33:00        0.192361
         C536548          536401     22767               -2              2       9.95 2010-12-01 11:21:00 2010-12-01 14:33:00        0.133333
         C536548          536522     

In [7]:
product_summary = (
    df.assign(
        PositiveQuantity=df["Quantity"].clip(lower=0),
        NegativeQuantity=df["Quantity"].clip(upper=0),
    )
    .groupby("StockCode")
    .agg(
        PositiveQuantity=("PositiveQuantity", "sum"),
        NegativeQuantity=("NegativeQuantity", "sum"),
    )
)

product_summary["NegativeQuantityRatio"] = (
    product_summary["NegativeQuantity"].abs()
    / product_summary["PositiveQuantity"]
)

print("\n--- Products With Highest Negative-to-Positive Quantity Ratios ---")

print(
    product_summary[
        product_summary["PositiveQuantity"] > 0
    ]
    .sort_values("NegativeQuantityRatio", ascending=False)
    .head(20)
    .to_string()
)


--- Products With Highest Negative-to-Positive Quantity Ratios ---
           PositiveQuantity  NegativeQuantity  NegativeQuantityRatio
StockCode                                                           
79323P                    4             -2011             502.750000
79323B                    8             -1679             209.875000
84614A                    2              -390             195.000000
84613C                    1              -173             173.000000
35610A                   10             -1277             127.700000
22351                    13             -1400             107.692308
35609A                    1               -51              51.000000
35610C                   25             -1121              44.840000
84743C                    1               -42              42.000000
84613A                    6              -230              38.333333
84569C                    1               -37              37.000000
21839                     1        

In [8]:
problem_products = [
    "79323P",
    "79323B",
    "84614A",
    "84613C",
    "35610A",
]

print("\n--- Problem Product Investigation ---")

print(
    df[
        df["StockCode"].astype(str).str.strip().isin(problem_products)
    ][
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
        ]
    ]
    .sort_values(["StockCode", "InvoiceDate"])
    .to_string(index=False)
)


--- Problem Product Investigation ---
InvoiceNo StockCode                        Description  Quantity         InvoiceDate
   536544    35610A      PINK CHRISTMAS FLOCK DROPLET          1 2010-12-01 14:32:00
   538349    35610A      PINK CHRISTMAS FLOCK DROPLET          1 2010-12-10 14:59:00
   538993    35610A      PINK CHRISTMAS FLOCK DROPLET          8 2010-12-15 12:01:00
   546130    35610A                        thrown away     -1277 2011-03-09 15:08:00
  C540307    79323B                BLACK CHERRY LIGHTS        -4 2011-01-06 12:58:00
  C543745    79323B                BLACK CHERRY LIGHTS        -4 2011-02-11 13:46:00
   543747    79323B                BLACK CHERRY LIGHTS         8 2011-02-11 13:53:00
   573591    79323B             Unsaleable, destroyed.     -1671 2011-10-31 15:15:00
  C540634    79323P                 PINK CHERRY LIGHTS        -4 2011-01-10 12:02:00
   540825    79323P                 PINK CHERRY LIGHTS         4 2011-01-11 13:54:00
  C543744    79323P       

In [10]:
negative_transactions = df[
    df["Quantity"] < 0
].copy()

print("\n--- Negative Transactions by Description ---")

print(
    negative_transactions["Description"]
    .fillna("<MISSING>")
    .value_counts()
    .head(30)
    .to_string()
)

print("\n--- Negative Transactions Without Cancellation Invoice ---")

print(
    negative_transactions[
        ~negative_transactions["InvoiceNo"]
        .astype(str)
        .str.startswith("C")
    ][
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
        ]
    ]
    .head(50)
    .to_string(index=False)
)


--- Negative Transactions by Description ---
Description
<MISSING>                              862
Manual                                 244
REGENCY CAKESTAND 3 TIER               181
POSTAGE                                126
check                                  120
JAM MAKING SET WITH JARS                87
Discount                                77
SET OF 3 CAKE TINS PANTRY DESIGN        74
SAMPLES                                 61
STRAWBERRY CERAMIC TRINKET BOX          55
ROSES REGENCY TEACUP AND SAUCER         54
RECIPE BOX PANTRY YELLOW DESIGN         47
damages                                 45
WOOD 2 DRAWER CABINET WHITE FINISH      45
JUMBO BAG RED RETROSPOT                 44
LUNCH BAG RED RETROSPOT                 44
RED RETROSPOT CAKE STAND                42
WHITE HANGING HEART T-LIGHT HOLDER      42
GREEN REGENCY TEACUP AND SAUCER         42
damaged                                 42
?                                       41
SMALL GLASS HEART TRINKET POT          

In [12]:
# 4. Inspect Cancellations
#
# This is exploratory only. The source does not provide a reliable
# cancellation-to-original-sale relationship, so these matches are
# not used in the production cleaning pipeline.

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["StockCode"] = df["StockCode"].astype(str).str.strip()

sales = df[
    df["Quantity"] > 0
].copy()

cancellations = df[
    df["InvoiceNo"]
    .astype(str)
    .str.startswith("C")
].copy()

cancellations["CancelledQuantity"] = cancellations["Quantity"].abs()

print("--- Cancellation Investigation ---")
print(f"Sales rows: {len(sales):,}")
print(f"Cancellation rows: {len(cancellations):,}")

sales = sales.reset_index().rename(
    columns={"index": "SaleRowID"}
)

cancellations = cancellations.reset_index().rename(
    columns={"index": "CancellationRowID"}
)

sales = sales.sort_values("InvoiceDate")
cancellations = cancellations.sort_values("InvoiceDate")

used_sales = set()
matches = []

for _, cancellation in cancellations.iterrows():

    candidates = sales[
        (sales["StockCode"] == cancellation["StockCode"])
        & (sales["UnitPrice"] == cancellation["UnitPrice"])
        & (sales["Quantity"] == cancellation["CancelledQuantity"])
        & (sales["InvoiceDate"] < cancellation["InvoiceDate"])
        & (~sales["SaleRowID"].isin(used_sales))
    ]

    if candidates.empty:
        continue

    candidates = candidates.copy()

    candidates["TimeDifference"] = (
        cancellation["InvoiceDate"]
        - candidates["InvoiceDate"]
    )

    best = candidates.loc[
        candidates["TimeDifference"].idxmin()
    ]

    used_sales.add(best["SaleRowID"])

    matches.append(
        {
            "CancellationRowID": cancellation["CancellationRowID"],
            "InvoiceNo_cancel": cancellation["InvoiceNo"],
            "InvoiceNo_sale": best["InvoiceNo"],
            "StockCode": cancellation["StockCode"],
            "Quantity_cancel": cancellation["Quantity"],
            "Quantity_sale": best["Quantity"],
            "UnitPrice": cancellation["UnitPrice"],
            "InvoiceDate_sale": best["InvoiceDate"],
            "InvoiceDate_cancel": cancellation["InvoiceDate"],
            "DaysDifference": (
                cancellation["InvoiceDate"]
                - best["InvoiceDate"]
            ).total_seconds() / 86400,
        }
    )

matches = pd.DataFrame(matches)

print("\n--- Matching Results ---")
print(f"Matched cancellation rows: {len(matches):,}")
print(
    f"Unmatched cancellation rows: "
    f"{len(cancellations) - len(matches):,}"
)

if not matches.empty:

    match_columns = [
        "InvoiceNo_cancel",
        "InvoiceNo_sale",
        "StockCode",
        "Quantity_cancel",
        "Quantity_sale",
        "UnitPrice",
        "InvoiceDate_sale",
        "InvoiceDate_cancel",
        "DaysDifference",
    ]

    print("\n--- Example Matches ---")
    print(
        matches[match_columns]
        .head(20)
        .to_string(index=False)
    )

    print("\n--- Largest Time Differences ---")
    print(
        matches
        .sort_values("DaysDifference", ascending=False)
        [match_columns]
        .head(20)
        .to_string(index=False)
    )

    print("\n--- Time Difference Statistics ---")
    print(
        matches["DaysDifference"].describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
            ]
        )
    )

    print("\n--- Matches Within Time Windows ---")

    for days in [1, 7, 14, 30, 60, 90, 180, 365]:

        count = (
            matches["DaysDifference"] <= days
        ).sum()

        percentage = count / len(matches) * 100

        print(
            f"Within {days:>3} days: "
            f"{count:>5,} ({percentage:.1f}%)"
        )

--- Cancellation Investigation ---
Sales rows: 531,285
Cancellation rows: 9,288

--- Matching Results ---
Matched cancellation rows: 6,994
Unmatched cancellation rows: 2,294

--- Example Matches ---
InvoiceNo_cancel  InvoiceNo_sale StockCode  Quantity_cancel  Quantity_sale  UnitPrice    InvoiceDate_sale  InvoiceDate_cancel  DaysDifference
         C536506          536466     22960               -6              6       4.25 2010-12-01 12:23:00 2010-12-01 12:38:00        0.010417
         C536543          536522     22632               -1              1       2.10 2010-12-01 12:49:00 2010-12-01 14:30:00        0.070139
         C536548          536385     22168               -2              2       8.50 2010-12-01 09:56:00 2010-12-01 14:33:00        0.192361
         C536548          536401     22767               -2              2       9.95 2010-12-01 11:21:00 2010-12-01 14:33:00        0.133333
         C536548          536522     20914               -1              1       2.95 2010-